# Axon — Layer 1 Extraction Pipeline Demo

**From floor plan to structural graph.** This notebook walks through the four implemented stages of Axon's Layer 1 extraction pipeline:

| Stage | Module | What it does |
|-------|--------|--------------|
| 1 | **Parser** | PDF → raw spatial graph G₀ (PostScript extraction, Bézier sampling, vertex dedup) |
| 2 | **Tokenizer** | G₀ → enriched tokens (cross-modal vector↔raster fusion via TEF) |
| 3 | **Diffusion** | Tokens → refined graph G* (DDPM denoising with HDSE structural bias) |
| 4 | **Constraints** | G* → clean graph (differentiable SAT axioms + hard geometric projection) |

> **Note:** Models are randomly initialized (no trained checkpoints yet). This demo validates the forward pass and data flow, not prediction quality.

## 0. Setup

In [ ]:
# Upload the project to Colab
#
# Option A (default): Upload a zip of the repo.
#   1. Zip your local Axon folder:
#        cd /path/to/parent && zip -r Axon.zip Axon/ -x 'Axon/.venv/*' 'Axon/__pycache__/*'
#   2. Run this cell and upload the zip.
#
# Option B: Mount Google Drive if the repo is already there.
#   Uncomment the Drive section and comment out Option A.

import os

# ── Option A: upload zip ──────────────────────────────────────────
USE_ZIP_UPLOAD = True  # set False if using Drive instead

if USE_ZIP_UPLOAD:
    from google.colab import files as colab_files
    print("Upload Axon.zip (the zipped project root):")
    uploaded = colab_files.upload()
    if uploaded:
        zip_name = list(uploaded.keys())[0]
        !unzip -qo "{zip_name}" -d /content/
        # Handle top-level folder presence
        if os.path.isdir("/content/Axon"):
            %cd /content/Axon
        else:
            %cd /content

# ── Option B: Google Drive ─────────────────────────────────────────
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive/Axon   # adjust path as needed

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
# torch + torchvision come pre-installed on Colab, but we pin compatible versions
!pip install -q PyMuPDF>=1.24.0 timm>=1.0.0 pydantic>=2.6.0 scipy>=1.12.0 networkx>=3.2

# torch-geometric requires matching torch/CUDA versions
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {TORCH_VERSION}, CUDA {CUDA_VERSION}")

!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html 2>/dev/null || \
    pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cpu.html

print("Dependencies installed.")

In [ ]:
# Ensure the project root is on sys.path so `src.*` and `docs.*` imports work
import sys

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Create a Test Floor Plan PDF

We programmatically create a simple multi-room floor plan using PyMuPDF. This avoids needing to upload a real PDF for the demo — but you can substitute your own in Section 7.

In [ ]:
import fitz  # PyMuPDF
from pathlib import Path


def create_demo_floor_plan(path: str = "/tmp/demo_floor_plan.pdf") -> str:
    """Create a simple multi-room floor plan PDF.

    Layout (not to scale):
    +----------------------+------------+
    |                      |            |
    |     Living Room      |  Kitchen   |
    |                      |            |
    |                      +------------+
    |                      |            |
    +----------------------+  Bathroom  |
    |                      |            |
    |     Bedroom          |            |
    |                      |            |
    +----------------------+------------+
    """
    doc = fitz.open()
    page = doc.new_page(width=612, height=792)
    shape = page.new_shape()

    WALL_WIDTH = 2.0
    BLACK = (0, 0, 0)

    # All coordinates in PDF user units (72/inch)
    walls = [
        # Exterior walls
        (80, 100, 530, 100),   # top
        (530, 100, 530, 600),  # right
        (530, 600, 80, 600),   # bottom
        (80, 600, 80, 100),    # left

        # Vertical interior wall (separates left rooms from right rooms)
        (370, 100, 370, 600),

        # Horizontal interior wall (separates living room from bedroom)
        (80, 400, 370, 400),

        # Horizontal interior wall (separates kitchen from bathroom)
        (370, 350, 530, 350),
    ]

    for x0, y0, x1, y1 in walls:
        shape.draw_line(fitz.Point(x0, y0), fitz.Point(x1, y1))
        shape.finish(width=WALL_WIDTH, color=BLACK)

    # Add some decorative elements the parser should flag as low-confidence
    # Dimension lines (thin, dashed)
    shape.draw_line(fitz.Point(80, 85), fitz.Point(530, 85))
    shape.finish(width=0.25, color=(0.4, 0.4, 0.4), dashes="[4 2]")

    shape.draw_line(fitz.Point(65, 100), fitz.Point(65, 600))
    shape.finish(width=0.25, color=(0.4, 0.4, 0.4), dashes="[4 2]")

    shape.commit()
    doc.save(path)
    doc.close()
    return path


PDF_PATH = create_demo_floor_plan()
print(f"Created test PDF: {PDF_PATH}")

In [ ]:
# Visualize the PDF as a raster image
import matplotlib.pyplot as plt
import numpy as np

doc = fitz.open(PDF_PATH)
page = doc[0]
pix = page.get_pixmap(dpi=150)
img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
doc.close()

fig, ax = plt.subplots(1, 1, figsize=(8, 10))
ax.imshow(img)
ax.set_title("Input: Demo Floor Plan PDF")
ax.axis("off")
plt.tight_layout()
plt.show()

---

## 2. Stage 1 — Parser (PDF → RawGraph)

The parser reads PostScript operators from the PDF content stream, extracts path geometry, samples Bézier curves into polylines, deduplicates vertices via KD-tree, and computes per-edge wall confidence heuristics.

**Output:** `RawGraph` — nodes (N×2 coordinates), edges (E×2 index pairs), stroke properties, wall confidence scores.

In [ ]:
from src.parser import extract_paths_from_pdf, build_raw_graph, apply_filters
from src.pipeline.config import ParserConfig

parser_config = ParserConfig()

# Extract paths from PDF
pages = extract_paths_from_pdf(
    PDF_PATH,
    page_indices=[0],
    config=parser_config,
)
paths = pages.get(0, [])
print(f"Extracted {len(paths)} paths from page 0")

# Get page dimensions
doc = fitz.open(PDF_PATH)
page = doc[0]
page_width, page_height = float(page.rect.width), float(page.rect.height)
doc.close()
print(f"Page dimensions: {page_width} x {page_height} PDF units")

# Build the raw graph
raw_graph = build_raw_graph(
    paths,
    config=parser_config,
    page_width=page_width,
    page_height=page_height,
    page_index=0,
    source_path=PDF_PATH,
)

# Apply filters (stroke width, color heuristics)
raw_graph = apply_filters(raw_graph, config=parser_config)

print(f"\nRawGraph:")
print(f"  Nodes: {raw_graph.nodes.shape[0]}")
print(f"  Edges: {raw_graph.edges.shape[0]}")
print(f"  Wall confidence range: [{raw_graph.confidence_wall.min():.3f}, {raw_graph.confidence_wall.max():.3f}]")
print(f"  Stroke widths: {np.unique(raw_graph.stroke_widths)}")

In [ ]:
# Visualize the raw graph
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Left: all edges colored by wall confidence
ax = axes[0]
ax.set_title("RawGraph — edges colored by wall confidence")
ax.set_xlim(0, page_width)
ax.set_ylim(page_height, 0)  # PDF y-axis is top-down
ax.set_aspect("equal")

for i in range(raw_graph.edges.shape[0]):
    src, dst = raw_graph.edges[i]
    x = [raw_graph.nodes[src, 0], raw_graph.nodes[dst, 0]]
    y = [raw_graph.nodes[src, 1], raw_graph.nodes[dst, 1]]
    conf = raw_graph.confidence_wall[i]
    color = plt.cm.RdYlGn(conf)  # red=low, green=high
    lw = raw_graph.stroke_widths[i]
    ax.plot(x, y, color=color, linewidth=max(lw, 0.5), alpha=0.8)

sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=plt.Normalize(0, 1))
plt.colorbar(sm, ax=ax, label="Wall Confidence")

# Right: high-confidence edges only (walls)
ax = axes[1]
ax.set_title("RawGraph — high confidence edges only (>0.5)")
ax.set_xlim(0, page_width)
ax.set_ylim(page_height, 0)
ax.set_aspect("equal")

for i in range(raw_graph.edges.shape[0]):
    if raw_graph.confidence_wall[i] > 0.5:
        src, dst = raw_graph.edges[i]
        x = [raw_graph.nodes[src, 0], raw_graph.nodes[dst, 0]]
        y = [raw_graph.nodes[src, 1], raw_graph.nodes[dst, 1]]
        ax.plot(x, y, color="black", linewidth=1.5)

# Mark nodes
ax.scatter(raw_graph.nodes[:, 0], raw_graph.nodes[:, 1], c="red", s=20, zorder=5)

plt.tight_layout()
plt.show()

---

## 3. Stage 2 — Tokenizer (RawGraph → EnrichedTokenSequence)

The tokenizer converts graph edges into learned token embeddings with 2D positional encoding, optionally extracts multi-scale raster features via HRNet/Swin, and fuses them with bidirectional cross-attention (TEF).

**Output:** `EnrichedTokenSequence` — (B, N, 256) fused embeddings ready for the diffusion model.

In [ ]:
from src.tokenizer import Tokenizer, collate_graphs, render_pdf_page, preprocess_image
from src.pipeline.config import TokenizerConfig

tok_config = TokenizerConfig(
    raster_dpi=150,  # lower DPI for Colab memory
)

tokenizer = Tokenizer(config=tok_config, n_tef_layers=2)
tokenizer.to(DEVICE)
tokenizer.eval()

# Prepare graph features as batched tensors
features = collate_graphs([raw_graph])
features = {k: v.to(DEVICE) for k, v in features.items()}

print("Collated graph features:")
for k, v in features.items():
    print(f"  {k}: {v.shape} ({v.dtype})")

In [ ]:
# Run tokenizer in vector-only mode (no vision backbone download needed)
with torch.no_grad():
    enriched_vec = tokenizer(features, images=None)

print(f"\nEnrichedTokenSequence (vector-only):")
print(f"  token_embeddings: {enriched_vec.token_embeddings.shape}")
print(f"  attention_mask:   {enriched_vec.attention_mask.shape}")
print(f"  d_model:          {enriched_vec.d_model}")
print(f"  is_vector_only:   {enriched_vec.is_vector_only}")
print(f"  valid tokens:     {enriched_vec.attention_mask.sum().item():.0f}")

In [ ]:
# Optionally run with raster fusion (downloads HRNet weights ~80MB)
USE_RASTER = False  # Set True to test cross-modal fusion

if USE_RASTER:
    raster = render_pdf_page(PDF_PATH, page_index=0, dpi=tok_config.raster_dpi)
    images = preprocess_image(raster).to(DEVICE)
    print(f"Raster image tensor: {images.shape}")

    with torch.no_grad():
        enriched_fused = tokenizer(features, images=images)

    print(f"\nEnrichedTokenSequence (cross-modal):")
    print(f"  token_embeddings: {enriched_fused.token_embeddings.shape}")
    print(f"  is_vector_only:   {enriched_fused.is_vector_only}")
    print(f"  raster_features:  {enriched_fused.raster_features.shape if enriched_fused.raster_features is not None else None}")

# Use whichever enriched sequence we produced
enriched = enriched_fused if USE_RASTER else enriched_vec

In [ ]:
# Visualize token embedding space (PCA projection)
from sklearn.decomposition import PCA

embeddings = enriched.token_embeddings[0].cpu().numpy()  # (N, 256)
mask = enriched.attention_mask[0].cpu().numpy().astype(bool)
valid_emb = embeddings[mask]

if valid_emb.shape[0] >= 3:
    pca = PCA(n_components=3)
    proj = pca.fit_transform(valid_emb)

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    sc = ax.scatter(proj[:, 0], proj[:, 1], c=proj[:, 2], cmap="viridis", s=40, alpha=0.8)
    plt.colorbar(sc, ax=ax, label="PC3")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Token Embeddings — PCA projection ({valid_emb.shape[0]} tokens, d={enriched.d_model})")
    plt.tight_layout()
    plt.show()

    print(f"PCA explained variance: {pca.explained_variance_ratio_[:3].sum():.1%} (top 3 components)")
else:
    print(f"Only {valid_emb.shape[0]} valid tokens — skipping PCA visualization")

---

## 4. Stage 3 — Diffusion Engine (EnrichedTokenSequence → RefinedStructuralGraph)

The graph diffusion model takes enriched token embeddings as conditioning context and iteratively denoises a random graph through DDIM sampling. The transformer backbone uses HDSE (Hierarchical Distance Structural Encoding) for structural inductive bias.

**Output:** `RefinedStructuralGraph` — denoised node positions and adjacency logits.

> With untrained weights, the output is essentially random — but this validates the full sampling loop.

In [ ]:
from src.diffusion import GraphDiffusionModel
from src.pipeline.config import DiffusionConfig

diff_config = DiffusionConfig(
    n_layers=4,             # fewer layers for speed (default 12)
    timesteps_train=200,    # fewer timesteps for demo (default 1000)
    timesteps_inference=20, # fewer DDIM steps (default 50)
    max_nodes=64,           # cap nodes for memory (default 512)
)

diffusion_model = GraphDiffusionModel(config=diff_config)
diffusion_model.to(DEVICE)
diffusion_model.eval()

num_params = sum(p.numel() for p in diffusion_model.parameters())
print(f"Diffusion model: {num_params:,} parameters")

In [ ]:
# Run DDIM sampling
num_nodes = min(raw_graph.nodes.shape[0], diff_config.max_nodes)
context = enriched.token_embeddings.to(DEVICE)
context_mask = enriched.attention_mask.to(DEVICE)

print(f"Sampling {num_nodes} nodes with {diff_config.timesteps_inference} DDIM steps...")

with torch.no_grad():
    refined = diffusion_model.sample(
        num_nodes=num_nodes,
        batch_size=1,
        context=context,
        context_mask=context_mask,
        device=DEVICE,
    )

print(f"\nRefinedStructuralGraph:")
print(f"  node_positions:   {refined.node_positions.shape}")
print(f"  adjacency_logits: {refined.adjacency_logits.shape}")
print(f"  node_mask:        {refined.node_mask.shape} ({refined.node_mask.sum().item():.0f} valid)")
print(f"  edge_index:       {refined.edge_index.shape}")
print(f"  denoising_step:   {refined.denoising_step}")

In [ ]:
# Visualize diffusion output
positions = refined.node_positions[0].cpu().numpy()  # (N, 2) in [0, 1]
adj_logits = refined.adjacency_logits[0].cpu()
adj_probs = torch.sigmoid(adj_logits).numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Node positions
ax = axes[0]
ax.scatter(positions[:, 0], positions[:, 1], c="steelblue", s=30)
ax.set_title(f"Diffusion output — node positions ({num_nodes} nodes)")
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(1.1, -0.1)
ax.set_aspect("equal")
ax.set_xlabel("x (normalized)")
ax.set_ylabel("y (normalized)")

# Adjacency logits heatmap
ax = axes[1]
im = ax.imshow(adj_probs, cmap="Blues", vmin=0, vmax=1)
ax.set_title("Adjacency probabilities (sigmoid of logits)")
ax.set_xlabel("Node j")
ax.set_ylabel("Node i")
plt.colorbar(im, ax=ax)

# Thresholded graph
ax = axes[2]
ax.set_title("Thresholded graph (p > 0.5)")
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(1.1, -0.1)
ax.set_aspect("equal")

binary = (adj_probs > 0.5).astype(float)
np.fill_diagonal(binary, 0)
edges_i, edges_j = np.where(np.triu(binary) > 0)
for i, j in zip(edges_i, edges_j):
    ax.plot(
        [positions[i, 0], positions[j, 0]],
        [positions[i, 1], positions[j, 1]],
        color="steelblue", linewidth=1.0, alpha=0.6,
    )
ax.scatter(positions[:, 0], positions[:, 1], c="red", s=20, zorder=5)
ax.set_xlabel("x (normalized)")

plt.tight_layout()
plt.show()

print(f"Edges above threshold: {len(edges_i)}")

---

## 5. Stage 4 — Constraint Solver (RefinedStructuralGraph → ConstraintGradients)

The NeSy SAT solver applies four differentiable geometric axioms:

| Axiom | ID | What it enforces |
|-------|----|------------------|
| Orthogonal Integrity | C-001 | 90°/180° wall angles |
| Parallel Pair Constancy | C-002 | Uniform wall thickness |
| Junction Closure | C-003 | No dangling endpoints |
| Spatial Non-Intersection | C-004 | No crossing walls |

During training these provide soft losses. At inference the geometric projector also applies **hard snapping** (orthogonal correction, dangling endpoint closure, intersection resolution).

**Output:** `ConstraintGradients` — per-axiom losses, violation masks, and optionally projected geometry.

In [ ]:
from src.constraints import ConstraintSolver
from src.pipeline.config import ConstraintConfig

constraint_config = ConstraintConfig()
solver = ConstraintSolver(config=constraint_config)
solver.to(DEVICE)
solver.eval()

with torch.no_grad():
    constraints = solver(
        node_positions=refined.node_positions,
        adjacency=refined.adjacency_logits,
        edge_index=refined.edge_index,
        node_mask=refined.node_mask,
        denoising_step=0,
        total_steps=refined.total_steps,
        is_inference=True,
    )

print("Constraint results:")
print(f"  Total loss: {constraints.total_loss.item():.4f}")
print(f"  Projected positions available: {constraints.projected_positions is not None}")
print(f"  Projected adjacency available: {constraints.projected_adjacency is not None}")

print(f"\nPer-axiom losses:")
for result in constraints.axiom_results:
    name = result.axiom_name if hasattr(result, 'axiom_name') else '?'
    print(f"  {name}: loss={result.loss.item():.4f}, weight={result.weight.item():.4f}")

In [ ]:
# Visualize before vs after constraint projection
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (title, pos_tensor) in enumerate([
    ("Before constraints", refined.node_positions),
    ("After constraint projection", constraints.projected_positions if constraints.projected_positions is not None else refined.node_positions),
]):
    ax = axes[ax_idx]
    pos = pos_tensor[0].cpu().numpy()

    # Use projected adjacency if available, else threshold logits
    if ax_idx == 1 and constraints.projected_adjacency is not None:
        adj = constraints.projected_adjacency[0].cpu().numpy()
    else:
        adj = torch.sigmoid(refined.adjacency_logits[0].cpu()).numpy()

    binary = (adj > 0.5).astype(float)
    np.fill_diagonal(binary, 0)
    ei, ej = np.where(np.triu(binary) > 0)

    for i, j in zip(ei, ej):
        ax.plot(
            [pos[i, 0], pos[j, 0]],
            [pos[i, 1], pos[j, 1]],
            color="steelblue", linewidth=1.2,
        )
    ax.scatter(pos[:, 0], pos[:, 1], c="red", s=25, zorder=5)

    ax.set_title(title)
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(1.1, -0.1)
    ax.set_aspect("equal")
    ax.set_xlabel("x (normalized)")

plt.tight_layout()
plt.show()

---

## 6. Full Pipeline — End to End

The `Layer1Pipeline` class chains all four stages with a single `extract()` call and returns a `FinalizedGraph` ready for downstream serialization.

In [ ]:
from src.pipeline import Layer1Pipeline, AxonConfig
from src.pipeline.config import DiffusionConfig, TokenizerConfig

# Configure with smaller models for Colab
config = AxonConfig(
    tokenizer=TokenizerConfig(raster_dpi=150),
    diffusion=DiffusionConfig(
        n_layers=4,
        timesteps_train=200,
        timesteps_inference=20,
        max_nodes=64,
    ),
)

pipeline = Layer1Pipeline(config=config, device=DEVICE)
print("Pipeline initialized.")

In [ ]:
# Run full extraction (vector-only mode)
result = pipeline.extract(PDF_PATH, page_index=0, use_raster=False)

print(f"FinalizedGraph:")
print(f"  Nodes:           {result.nodes.shape}")
print(f"  Edges:           {result.edges.shape}")
print(f"  Wall segments:   {len(result.wall_segments)}")
print(f"  Rooms:           {len(result.rooms)}")
print(f"  Openings:        {len(result.openings)}")
print(f"  Betti-0:         {result.betti_0} (connected components)")
print(f"  Betti-1:         {result.betti_1} (enclosed loops)")
print(f"  Page:            {result.page_width} x {result.page_height}")
print(f"  Wall height:     {result.assumed_wall_height} mm")

In [ ]:
# Visualize the finalized graph
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

# Draw wall segments
for seg in result.wall_segments:
    ax.plot(
        [seg.start_coord[0], seg.end_coord[0]],
        [seg.start_coord[1], seg.end_coord[1]],
        color="black",
        linewidth=max(seg.thickness / 3, 0.8),
        solid_capstyle="round",
    )

# Draw nodes
if result.nodes.shape[0] > 0:
    ax.scatter(
        result.nodes[:, 0], result.nodes[:, 1],
        c="red", s=30, zorder=5, label=f"{result.nodes.shape[0]} junctions",
    )

ax.set_xlim(0, result.page_width)
ax.set_ylim(result.page_height, 0)
ax.set_aspect("equal")
ax.set_title(
    f"FinalizedGraph — {len(result.wall_segments)} walls, "
    f"\u03b2\u2080={result.betti_0}, \u03b2\u2081={result.betti_1}"
)
ax.legend()
ax.set_xlabel("x (PDF units)")
ax.set_ylabel("y (PDF units)")
plt.tight_layout()
plt.show()

In [ ]:
# Print wall segment details
print(f"{'Edge':>4}  {'Length':>8}  {'Thick':>6}  {'Angle\u00b0':>7}  {'Type'}")
print("-" * 45)
for seg in result.wall_segments[:20]:  # first 20
    angle_deg = np.degrees(seg.angle)
    print(
        f"{seg.edge_id:4d}  {seg.length:8.1f}  {seg.thickness:6.1f}  "
        f"{angle_deg:7.1f}  {seg.wall_type.value}"
    )

if len(result.wall_segments) > 20:
    print(f"  ... ({len(result.wall_segments) - 20} more)")

---

## 7. Upload Your Own PDF

Replace the demo floor plan with a real architectural PDF.

In [ ]:
# Upload a PDF from your local machine
from google.colab import files

print("Upload a floor plan PDF (or skip this cell to use the demo):")
try:
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        user_pdf = f"/tmp/{filename}"
        with open(user_pdf, "wb") as f:
            f.write(uploaded[filename])
        print(f"Saved to {user_pdf}")

        # Run the pipeline on the uploaded PDF
        user_result = pipeline.extract(user_pdf, page_index=0, use_raster=False)
        print(f"\nExtracted: {user_result.nodes.shape[0]} nodes, {len(user_result.wall_segments)} walls")
        print(f"Betti: \u03b2\u2080={user_result.betti_0}, \u03b2\u2081={user_result.betti_1}")
except ImportError:
    print("Not running in Colab — skipping upload widget.")
    print("Set user_pdf = '/path/to/your.pdf' and re-run.")

---

## 8. Inspect Model Architecture

Summary of model sizes and architecture for each stage.

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print(f"{'Module':<25} {'Total Params':>15} {'Trainable':>15}")
print("=" * 58)
for name, module in [
    ("Tokenizer", pipeline.tokenizer),
    ("Diffusion Model", pipeline.diffusion_model),
    ("Constraint Solver", pipeline.constraint_solver),
]:
    total, trainable = count_params(module)
    print(f"{name:<25} {total:>15,} {trainable:>15,}")

grand_total = sum(count_params(m)[0] for m in [
    pipeline.tokenizer, pipeline.diffusion_model, pipeline.constraint_solver
])
print("=" * 58)
print(f"{'TOTAL':<25} {grand_total:>15,}")

---

## 9. Forward Diffusion Visualization

Show how the forward diffusion process progressively corrupts a clean graph.

In [ ]:
from src.diffusion import ForwardDiffusion, DiffusionScheduler

# Create a clean grid graph to visualize corruption
grid_size = 5
coords = []
for i in range(grid_size):
    for j in range(grid_size):
        coords.append([i / (grid_size - 1), j / (grid_size - 1)])

x_0 = torch.tensor(coords, dtype=torch.float32).unsqueeze(0)  # (1, 25, 2)
N = x_0.shape[1]

# Build adjacency for a grid
adj_0 = torch.zeros(1, N, N)
for i in range(grid_size):
    for j in range(grid_size):
        idx = i * grid_size + j
        if j + 1 < grid_size:
            adj_0[0, idx, idx + 1] = 1
            adj_0[0, idx + 1, idx] = 1
        if i + 1 < grid_size:
            adj_0[0, idx, idx + grid_size] = 1
            adj_0[0, idx + grid_size, idx] = 1

scheduler = DiffusionScheduler(num_timesteps=200, schedule_type="cosine")
forward = ForwardDiffusion(scheduler)

timesteps_to_show = [0, 10, 50, 100, 150, 199]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(4 * len(timesteps_to_show), 4))

for ax_idx, t in enumerate(timesteps_to_show):
    ax = axes[ax_idx]

    if t == 0:
        pos = x_0[0].numpy()
        adj = adj_0[0].numpy()
    else:
        t_tensor = torch.tensor([t], dtype=torch.long)
        out = forward(x_0, adj_0, t_tensor)
        pos = out.x_t[0].numpy()
        adj = out.a_t[0].numpy()

    # Draw edges
    binary = (adj > 0.5).astype(float)
    np.fill_diagonal(binary, 0)
    ei, ej = np.where(np.triu(binary) > 0)
    for i, j in zip(ei, ej):
        ax.plot(
            [pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]],
            color="steelblue", linewidth=0.8, alpha=0.5,
        )
    ax.scatter(pos[:, 0], pos[:, 1], c="red", s=15, zorder=5)

    noise_level = 1 - scheduler.alphas_cumprod[min(t, len(scheduler.alphas_cumprod) - 1)].item()
    ax.set_title(f"t={t}\nnoise={noise_level:.2f}", fontsize=10)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_aspect("equal")
    ax.axis("off")

plt.suptitle("Forward Diffusion: progressive corruption of a clean grid graph", fontsize=13)
plt.tight_layout()
plt.show()

---

## 10. Constraint Axiom Deep Dive

Evaluate each axiom individually on the diffusion output.

In [ ]:
from src.constraints import (
    OrthogonalIntegrityAxiom,
    ParallelPairConstancyAxiom,
    JunctionClosureAxiom,
    SpatialNonIntersectionAxiom,
    edges_from_adjacency,
)

# Prepare inputs
pos = refined.node_positions.to(DEVICE)
adj = torch.sigmoid(refined.adjacency_logits).to(DEVICE)
edge_idx = edges_from_adjacency(adj[0], threshold=0.5)
mask = refined.node_mask.to(DEVICE)

axioms = [
    ("C-001 Orthogonal Integrity", OrthogonalIntegrityAxiom()),
    ("C-002 Parallel Pair Constancy", ParallelPairConstancyAxiom()),
    ("C-003 Junction Closure", JunctionClosureAxiom()),
    ("C-004 Spatial Non-Intersection", SpatialNonIntersectionAxiom()),
]

print(f"{'Axiom':<35} {'Loss':>10} {'Violations':>12}")
print("=" * 60)

for name, axiom in axioms:
    axiom.to(DEVICE)
    with torch.no_grad():
        result = axiom(
            node_positions=pos,
            adjacency=adj,
            edge_index=edge_idx,
            node_mask=mask,
        )
    n_violations = result.violation_mask.sum().item() if result.violation_mask is not None else 0
    print(f"{name:<35} {result.loss.item():>10.4f} {n_violations:>12.0f}")

---

## Summary

This notebook demonstrated the complete Layer 1 forward pass:

```
PDF  --[Parser]-->  RawGraph  --[Tokenizer]-->  EnrichedTokenSequence
         Stage 1                    Stage 2

     --[Diffusion]-->  RefinedStructuralGraph  --[Constraints]-->  FinalizedGraph
         Stage 3                                    Stage 4
```

**Next steps:**
- Train the models (MPM pre-training -> SFT -> GRPO curriculum)
- Implement Layer 2 (Knowledge Graph, DRL panelizer, BIM export)
- Evaluate on real architectural PDFs with ground truth